# NOVAPAY: FRAUDULENT TRANSACTION DETECTION FOR DIGITAL MONEY TRANSFER

## AUTHOR: INIOLUWA PIRISOLA

### DATA EXPLORATION

In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("/content/sample_data/nova_pay_combined.csv")

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df.head(5)

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [6]:
df.shape

(11400, 26)

In [7]:
df.isnull().sum()

,0
transaction_id,0
customer_id,0
timestamp,29
home_country,0
source_currency,0
dest_currency,0
channel,0
amount_src,0
amount_usd,305
fee,295


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  object 
 1   customer_id                11400 non-null  object 
 2   timestamp                  11371 non-null  object 
 3   home_country               11400 non-null  object 
 4   source_currency            11400 non-null  object 
 5   dest_currency              11400 non-null  object 
 6   channel                    11400 non-null  object 
 7   amount_src                 11400 non-null  object 
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  object 
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  obj

In [9]:
df["is_fraud"].value_counts(normalize=True)

,proportion
is_fraud,
0,0.912544
1,0.087456


### DATA CLEANING

Converting timestamp from object to datetime,
Converting amount_src from object to float

In [10]:
#convert timestamp to datetime
df['timestamp']=pd.to_datetime(df['timestamp'], errors='coerce')

In [11]:
#convert amount_srt to float
df['amount_src']=pd.to_numeric(df['amount_src'], errors='coerce')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11400 non-null  object             
 1   customer_id                11400 non-null  object             
 2   timestamp                  11339 non-null  datetime64[ns, UTC]
 3   home_country               11400 non-null  object             
 4   source_currency            11400 non-null  object             
 5   dest_currency              11400 non-null  object             
 6   channel                    11400 non-null  object             
 7   amount_src                 11396 non-null  float64            
 8   amount_usd                 11095 non-null  float64            
 9   fee                        11105 non-null  float64            
 10  exchange_rate_src_to_dest  11400 non-null  float64            
 11  de

Filling in missing values for targeted columns

Filling in missing values for amount_usd. This calculates exchange rates per currency by selecting non-missing rows in amount_usd and grouping by source currency, then computing the mean of amount_usd/amount_src for each currency and saving the results to a dictionary for easy lookup. The missing values in amount_src are filled by multiplying the values derived for the exchange rate and the source_currency column

In [13]:
#Filling in missing values for amount_usd
exchange_rates = df[df['amount_usd'].notna()].groupby('source_currency').apply(
    lambda x: (x['amount_usd']/x['amount_src']).mean()).to_dict()

/tmp/ipykernel_8213/1668463050.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  exchange_rates = df[df['amount_usd'].notna()].groupby('source_currency').apply(


In [14]:
exchange_rates

{'CAD': 0.7216095926871465,
 'GBP': 1.223441221648679,
 'USD': 0.9838730321259439}

In [15]:
df['amount_usd'] = df.apply(lambda row: row['amount_usd']
                            if pd.notna(row['amount_usd']) else row['amount_src']*exchange_rates.get(row['source_currency'], 1),axis=1)

In [16]:
# To view rows where amount_src is null, filter rows
null_rows = df[df['amount_src'].isnull()]

# Display results
print(null_rows)

# Optional: count how many
print("Number of null rows in amount_src:", null_rows.shape[0])

                            transaction_id  \
4812  95df4052-29be-470f-87f9-61e5173370b2   
8036  5569192e-3bf1-47e3-9723-d63c3dd30cad   
8878  19a415c4-05b3-40b5-ad6d-e7d34247e050   
9444  99a53933-5876-47df-ad3e-7976b48c01b4   

                               customer_id                        timestamp  \
4812  7041b9c1-3719-4ca8-9a6b-811b47cea6c0 2024-03-12 08:48:06.468549+00:00   
8036  66c71a44-d07c-4a20-a576-0a377193bf6b 2025-03-09 12:04:34.468549+00:00   
8878  6d0d9b27-fa26-45f8-93b1-2df29d182d9c 2025-06-04 08:02:23.468549+00:00   
9444  6d0d9b27-fa26-45f8-93b1-2df29d182d9c 2025-08-06 15:30:28.468549+00:00   

     home_country source_currency dest_currency channel  amount_src  \
4812           UK             GBP           CAD  mobile         NaN   
8036           US             USD           USD     web         NaN   
8878           US             USD           INR     ATM         NaN   
9444           US             USD           CAD  mobile         NaN   

      amount_usd 

In [17]:
df.iloc[4811:4815]

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
4811,db820cfc-7251-4f26-9ace-556250658e43,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2024-03-12 05:52:08.468549+00:00,US,USD,INR,mobile,439.16,439.16,7.86,...,0.129,enhanced,257,0.894,0,0.257,0,0,0.0,0
4812,95df4052-29be-470f-87f9-61e5173370b2,7041b9c1-3719-4ca8-9a6b-811b47cea6c0,2024-03-12 08:48:06.468549+00:00,UK,GBP,CAD,mobile,NaN,12498.57,-1.00,...,1.200,standard,4,-0.100,0,0.407,-1,0,0.0,0
4813,d4002b56-f3f5-426b-95f9-1191a6dc48c0,95de1ea4-0f12-4380-b781-16b6a490bebd,2024-03-12 12:26:57.468549+00:00,US,USD,USD,mobile,197.66,197.66,3.10,...,0.283,standard,1058,0.923,0,0.139,0,0,0.0,0
4814,4dd8f93c-326c-446b-b2c5-6f81701780d0,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2024-03-12 12:46:39.468549+00:00,US,USD,GBP,mobile,170.62,170.62,3.09,...,0.373,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [18]:
# Dropping the rows where amount_src is null since the values are too high and will cause an imbalance
df.dropna(subset=['amount_src'], inplace=True)

In [19]:
df.shape

(11396, 26)

In [20]:
# Filling in missing values for the fee column.
# This is done based on the median for the channel feature since fee is dependent on channel
if 'fee' in df.columns:
  if 'channel' in df.columns:
        df['fee'] = df.groupby('channel')['fee']\
              .transform(lambda x: x.fillna(x.median()))
        df['fee'] = df['fee'].fillna(df['fee'].median())

In [21]:
# Filling in the ip_country values
# This is done by using the corresponding home country
if 'ip_country' in df.columns and 'home_country' in df.columns:
  df['ip_country'] = df['ip_country'].fillna(df['home_country'])

In [22]:
# Filling in the missing values in kyc_tier with the mode
if 'kyc_tier' in df.columns:
  mode_kyc = df['kyc_tier'].mode().iloc[0] if not df['kyc_tier'].mode().empty else 'standard'
  df['kyc_tier'] = df['kyc_tier'].fillna(mode_kyc)

In [24]:
# Filling in the missing values in device_trust_score using the group's median value
if 'device_trust_score' in df.columns:
    if {'new_device','kyc_tier'}.issubset(df.columns):
        df['device_trust_score'] = df.groupby(['new_device','kyc_tier'])['device_trust_score'].transform(lambda x: x.fillna(x.median()))
    df['device_trust_score'] = df['device_trust_score'].fillna(df['device_trust_score'].median())

In [25]:
df.isnull().sum()

,0
transaction_id,0
customer_id,0
timestamp,61
home_country,0
source_currency,0
dest_currency,0
channel,0
amount_src,0
amount_usd,0
fee,0


Dropping rows with missing values where replacements cannot be applied

In [26]:
df.dropna(inplace= True)

In [27]:
df.isnull().sum()

,0
transaction_id,0
customer_id,0
timestamp,0
home_country,0
source_currency,0
dest_currency,0
channel,0
amount_src,0
amount_usd,0
fee,0


Dropping duplicated rows - 194 rows were dropped

In [30]:
# Checking for duplicated rown and dropping duplicates
df.duplicated().sum()

np.int64(0)

In [29]:
df.drop_duplicates(inplace=True)

### Sanity Checks for the dataset

This section lists essential sanity checks to validate the dataset after cleaning and imputation

1. Check for invalid numeric values, including negative values in monetary, risk, trust or velocity fields and validate the user age in days is not negative
2. Verify currency-related logic and ensure there are no negative values and that the derived exchange rates fall within reasonable range
3. Validate timestamp integrity and confirm no transaction timestamps occur in the future
4. Review location consistency to ensure the counts in location_mismatch was generated corrrectly and the country features contain plausible country codes
5. Check categorical column consistency to review unique values in channel, source_currency, dest_currency and kyc_tier to ensure there are no unexpected entries
6. Validate risk-score ranges ensuring all values fall within expected numeric range
7. Ensure standardization of formatting used in the features
8. Confirm fraud label integrity ensuring they contain only binary values
9. Validate velocity features

These checks ensure this dataset id thoroughly consistent before the feature engineering and modeling stage.

In [32]:
df.describe(include='all')

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
count,10836,10836,10836,10836,10836,10836,10836,10836.000000,10836.000000,10836.000000,...,10836.000000,10836,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000
unique,10836,1314,NaN,7,3,9,12,NaN,NaN,NaN,...,NaN,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,fdffeb16-192a-4483-9b1e-9928e23269c2,402cccc9-28de-45b3-9af7-cc5302aa1f93,NaN,US,USD,NGN,mobile,NaN,NaN,NaN,...,NaN,standard,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,1433,NaN,7533,7619,1400,6016,NaN,NaN,NaN,...,NaN,7733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,2024-05-03 11:39:15.294875648+00:00,NaN,NaN,NaN,NaN,437.384918,448.258985,97.071335,...,0.398331,NaN,392.088778,0.654070,0.050941,0.268593,0.475914,0.749169,0.045484,0.090993
min,NaN,NaN,2022-10-03 18:40:59.468549+00:00,NaN,NaN,NaN,NaN,-9997.160000,7.230000,-1.000000,...,0.004000,NaN,1.000000,-0.100000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,2023-08-15 06:58:52.468549120+00:00,NaN,NaN,NaN,NaN,90.910000,92.600000,2.390000,...,0.209000,NaN,147.000000,0.515000,0.000000,0.169000,0.000000,0.000000,0.000000,0.000000
50%,NaN,NaN,2024-05-09 14:30:08.521080064+00:00,NaN,NaN,NaN,NaN,159.080000,163.590000,3.510000,...,0.326000,NaN,272.000000,0.658000,0.000000,0.223000,0.000000,0.000000,0.000000,0.000000
75%,NaN,NaN,2025-01-29 08:52:54.047345408+00:00,NaN,NaN,NaN,NaN,295.485000,302.682500,5.560000,...,0.489250,NaN,661.000000,0.894000,0.000000,0.391000,0.000000,0.000000,0.050000,0.000000
max,NaN,NaN,2025-12-16 00:13:41.468549+00:00,NaN,NaN,NaN,NaN,11942.890000,12497.900000,9999.990000,...,1.200000,NaN,1095.000000,0.999000,2.000000,0.900000,8.000000,11.000000,0.250000,1.000000


In [33]:
# Count negative values in key numeric columns
negative_counts = {
    'amount_src': (df['amount_src'] < 0).sum(),
    'amount_usd': (df['amount_usd'] < 0).sum(),
    'fee': (df['fee'] < 0).sum(),
    'device_trust_score': (df['device_trust_score'] < 0).sum(),
    'txn_velocity_1h': (df['txn_velocity_1h'] < 0).sum(),
    'txn_velocity_24h': (df['txn_velocity_24h'] < 0).sum(),
    'risk_score_internal': (df['risk_score_internal'] < 0).sum()
}
negative_counts

{'amount_src': np.int64(97),
 'amount_usd': np.int64(0),
 'fee': np.int64(89),
 'device_trust_score': np.int64(186),
 'txn_velocity_1h': np.int64(186),
 'txn_velocity_24h': np.int64(0),
 'risk_score_internal': np.int64(0)}

In [35]:
df = df[(df['amount_src'] >=0) & (df['device_trust_score'] >= 0) & (df['fee'] >= 0) & (df['txn_velocity_1h'] >= 0)  ]

In [38]:
# Checking for correlation between cource amount and amount in usd
(df['amount_usd']/df['amount_src']).describe()

,0
count,10650.000000
mean,1.018225
std,0.136860
min,0.739788
25%,1.000000
50%,1.000000
75%,1.000000
max,1.250405


In [41]:
# Checking for any future dates and dropping if any
df[df['timestamp'] > pd.Timestamp.utcnow()]

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud


In [42]:
# Checking for the counts of mismatched location
df['location_mismatch'].value_counts()

,count
location_mismatch,
False,8884
True,1766


In [43]:
# Reviewing consistencies in categorical columns
df['kyc_tier'].unique()

array(['standard', 'enhanced', 'low', ' standard  ', 'standrd',
       ' enhanced  ', 'STANDARD', 'unknown', 'enhancd', ' low  ',
       'ENHANCED', 'LOW'], dtype=object)

In [44]:
df['source_currency'].unique()

array(['USD', 'CAD', 'GBP'], dtype=object)

In [45]:
df['dest_currency'].unique()

array(['CAD', 'MXN', 'CNY', 'EUR', 'INR', 'GBP', 'PHP', 'NGN', 'USD'],
      dtype=object)

In [47]:
df['channel'].unique()

array(['ATM', 'web', 'mobile', 'WEB', ' web  ', 'MOBILE', 'unknown',
       'mobille', ' mobile  ', 'weeb', 'ATm', ' ATM  '], dtype=object)

There appears to be a lot of inconsistencies including formatting and spellings and whitespaces so I standardize all of these inconsistencies

In [61]:
# Standardizing the channel feature
df['channel'] = df['channel'].replace({
    'web ': 'web',
    'web': 'web',
    'weeb': 'web',
    'WEB': 'web',
    ' web  ': 'web',
    'mobile ': 'mobile',
    ' mobile  ': 'mobile',
    'mobile': 'mobile',
    'mobille': 'mobile',
    'MOBILE': 'mobile',
    'ATM': 'atm',
    ' ATM  ': 'atm',
    'ATm': 'atm',
    'atm': 'atm',
    'ATM ': 'atm',
    'unknown': 'nan'
})

/tmp/ipykernel_8213/3307753974.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['channel'] = df['channel'].replace({


In [69]:
df['channel'].unique()

array(['atm', 'web', 'mobile', nan], dtype=object)

In [57]:
# Standardizing the kyc_tier feature
df['kyc_tier'] = df['kyc_tier'].replace({
    'standard ': 'standard',
    ' standard  ': 'standard',
    'standrd': 'standard',
    'STANDARD': 'standard',
    ' enhanced  ': 'enhanced',
    'ENHANCED ': 'enhanced',
    'enhancd': 'enhanced',
    'ENHANCED': 'enhanced',
    ' low  ': 'low',
    'LOW': 'low',
    'unknown': 'nan'
})





/tmp/ipykernel_8213/1513022809.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['kyc_tier'] = df['kyc_tier'].replace({


In [67]:
df['kyc_tier'].unique()

array(['standard', 'enhanced', 'low', nan], dtype=object)

In [68]:
df['channel'] = df['channel'].str.lower().str.strip()

/tmp/ipykernel_8213/1229850776.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['channel'] = df['channel'].str.lower().str.strip()


In [65]:
df['channel'] = df['channel'].replace({'nan': np.nan})

/tmp/ipykernel_8213/3457488447.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['channel'] = df['channel'].replace({'nan': np.nan})


In [66]:
df['kyc_tier'] = df['kyc_tier'].str.lower().str.strip()
df['kyc_tier'] = df['kyc_tier'].replace({'nan': np.nan})

/tmp/ipykernel_8213/4268530583.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['kyc_tier'] = df['kyc_tier'].str.lower().str.strip()
/tmp/ipykernel_8213/4268530583.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['kyc_tier'] = df['kyc_tier'].replace({'nan': np.nan})


In [70]:
df.isna().sum()

,0
transaction_id,0
customer_id,0
timestamp,0
home_country,0
source_currency,0
dest_currency,0
channel,36
amount_src,0
amount_usd,0
fee,0


In [72]:
# Dropping the missing values generated after standardization
df.dropna(inplace=True)

/tmp/ipykernel_8213/3319725727.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10591 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             10591 non-null  object             
 1   customer_id                10591 non-null  object             
 2   timestamp                  10591 non-null  datetime64[ns, UTC]
 3   home_country               10591 non-null  object             
 4   source_currency            10591 non-null  object             
 5   dest_currency              10591 non-null  object             
 6   channel                    10591 non-null  object             
 7   amount_src                 10591 non-null  float64            
 8   amount_usd                 10591 non-null  float64            
 9   fee                        10591 non-null  float64            
 10  exchange_rate_src_to_dest  10591 non-null  float64            
 11  device_